In [ ]:
from pathlib import Path
import os
import sys

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

os.chdir(root)
PROJECT_ROOT = root
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data: {PROCESSED_DATA_PATH}")


# Non Parametric Models

In [6]:
import pandas as pd

#### Import preprocessed data

In [7]:
PROCESSED_DATA_PATH = 'data/processed'

# 1 - non-scaled

X_train_non_scaled  = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_train_non_scaled.csv')
X_val_non_scaled    = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_val_non_scaled.csv')
X_test_non_scaled   = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_test_non_scaled.csv')

# 2 - robust scaled

X_train_scaled_robust = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_train_robust_scaled.csv')
X_val_scaled_robust   = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_val_robust_scaled.csv')
X_test_scaled_robust  = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_test_robust_scaled.csv')

# 3 - min-max scaled

X_train_scaled_minmax = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_train_minmax_scaled.csv')
X_val_scaled_minmax   = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_val_minmax_scaled.csv')
X_test_scaled_minmax  = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_test_minmax_scaled.csv')

# 4 - standard scaled
X_train_scaled_standard = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_train_standard_scaled.csv')
X_val_scaled_standard   = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_val_standard_scaled.csv')
X_test_scaled_standard  = pd.read_csv(f'{PROCESSED_DATA_PATH}/X_test_standard_scaled.csv')

# y
y_train = pd.read_csv(f'{PROCESSED_DATA_PATH}/y_train.csv')
y_val   = pd.read_csv(f'{PROCESSED_DATA_PATH}/y_val.csv')


X_train_full = pd.concat([X_train_non_scaled, X_val_non_scaled])
y_train_full = pd.concat([y_train, y_val])

print("\nAll versions loaded.")


All versions loaded.


In [8]:
'''
training.py
'''
import json
from datetime import datetime
import os
import shutil

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.utils.parallel import Parallel, delayed
import joblib
import wandb
import matplotlib.pyplot as plt


def refit_full(model, X_full: pd.DataFrame, y_full: pd.Series) -> None:
    """Refit best model on train+val combined before submission."""

    print("Refitting best model on full training data (train + val)...")
    model.fit(X_full, y_full)


def compute_metrics(y_real: pd.Series, y_pred: pd.Series) -> list[float]:
    accuracy = accuracy_score(y_real,y_pred)
    f1_macro =f1_score(y_real,y_pred, average='macro')
    precision_macro =precision_score(y_real,y_pred,  average='macro')
    recall_macro =recall_score(y_real,y_pred,  average='macro')
    classif_report = classification_report(y_real, y_pred)
    return [accuracy, f1_macro, precision_macro, recall_macro, classif_report]


def evaluate(base_name: str, model, X_val: pd.DataFrame, y_val: pd.Series,  best_params: dict|None = None) -> None:
    '''
    Evaluates the best model on the validation data and defines the experiment.
    '''
    print("Evaluating best model on unseen validation data...")

    y_pred = model.predict(X_val)

    define_experiment(base_name, compute_metrics(y_val, y_pred), y_val, y_pred)


import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.metrics import confusion_matrix

def plot_spatial_confusion(y_true, y_pred, base_name="Model"):
    # 1. Define physical coordinates (radius, angle_in_degrees) based on your image
    polar_coords = {
        0: (2, 0),   1: (2, 45),  2: (2, 90),  3: (2, 135), 4: (2, 180),
        5: (5, 0),   6: (5, 45),  7: (5, 90),  8: (5, 135), 9: (5, 180)
    }

    # Convert polar to Cartesian (x, y) coordinates for plotting
    coords = {}
    for label, (r, theta) in polar_coords.items():
        rad = np.radians(theta)
        coords[label] = (r * np.cos(rad), r * np.sin(rad))

    # Calculate standard confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=range(10))

    fig, ax = plt.subplots(figsize=(12, 8))

    # 2. Draw the Tracking Unit (centered at origin, pointing outward)
    tracking_unit = patches.Rectangle((-1.5, -1.5), 3, 1.5, color='black', zorder=5)
    ax.add_patch(tracking_unit)
    ax.text(0, -0.75, 'Tracking\nunit', color='white', ha='center', va='center', weight='bold', zorder=6)

    # 3. Draw the background dashed guidelines
    for label in [5, 6, 7, 8, 9]:
        x, y = coords[label]
        ax.plot([0, x], [0, y], color='black', linestyle='--', alpha=0.6, zorder=1)

    # 4. Plot the position nodes (0 through 9)
    for label, (x, y) in coords.items():
        ax.plot(x, y, 'o', markersize=25, color='white', markeredgecolor='black', zorder=4)
        ax.text(x, y, str(label), ha='center', va='center', fontsize=12, zorder=5)

    # 5. Draw the confusion arrows
    # Find the maximum off-diagonal value so we can scale the arrow thickness
    off_diag_mask = ~np.eye(cm.shape[0], dtype=bool)
    max_conf = np.max(cm[off_diag_mask]) if np.any(cm[off_diag_mask]) else 1

    for i in range(10):
        for j in range(10):
            # Only plot off-diagonal elements (errors) where count > 0
            if i != j and cm[i, j] > 0:
                count = cm[i, j]
                x1, y1 = coords[i] # True position
                x2, y2 = coords[j] # Predicted position (where it was mistakenly placed)

                # Scale arrow thickness (linewidth) and opacity (alpha) based on error frequency
                lw = max(1, (count / max_conf) * 5)
                alpha = min(0.3 + (count / max_conf) * 0.7, 1.0)

                # Create a curved arrow so bidirectional confusions (i->j and j->i) don't overlap
                arrow = patches.FancyArrowPatch(
                    (x1, y1), (x2, y2),
                    connectionstyle="arc3,rad=0.15",
                    arrowstyle="->,head_length=8,head_width=4",
                    color='red',
                    linewidth=lw,
                    alpha=alpha,
                    shrinkA=15, # Leaves a gap so the arrow doesn't overlap the circle text
                    shrinkB=15,
                    zorder=3
                )
                ax.add_patch(arrow)

    # 6. Formatting to match the physical aspect ratio

    ax.set_aspect('equal')
    ax.set_xlim(-6, 6)
    ax.set_ylim(-2, 6)
    ax.axis('off')
    plt.title(f'Spatial Error Map: {base_name}', fontsize=16, pad=15)
    plt.tight_layout()

    return fig


def define_experiment(base_name: str, metrics: list[float], y_val: pd.Series, y_pred: pd.Series,  best_params: dict|None = None) -> None:

    accuracy, f1_macro, precision_macro, recall_macro, classif_report = metrics
    experiment_results = {
        "model_name": base_name,
        "best_hyperparameters": best_params,
        "validation_metrics": {"accuracy": accuracy, "f1_macro": f1_macro}
    }

    print("Initializing Weights & Biases run...")
    wandb.init(
        project="AA1",
        entity="laura-rebollo-crespo-universitat-polit-cnica-de-catalunya",
        name=f"{base_name}_f1-{f1_macro:.4f}",
        config={"model_name": base_name, "best_params": best_params})

    fig, ax = plt.subplots(figsize=(10, 8))

    disp = ConfusionMatrixDisplay.from_predictions(
        y_val,
        y_pred,
        ax=ax,
        cmap='viridis',
        colorbar=False
    )
    plt.title(f'Confusion Matrix: {base_name}', fontsize=16, pad=15)
    plt.tight_layout()
    spatial = plot_spatial_confusion(y_val, y_pred, base_name)

    # LOG IT TO W&B
    wandb.log({
        "val_accuracy": accuracy,
        "val_f1_macro": f1_macro,
        "val_precision_macro": precision_macro,
        "val_recall_macro": recall_macro,
        "classification_report": wandb.Html(f"<pre>{classif_report}</pre>"),

        "scikit_learn_matrix": wandb.Image(fig),
        "spatial_confusion_matrix": wandb.Image(spatial)
    })


    plt.close(fig)
    plt.close(spatial)


def save(base_name:str, model) -> None:
    """
    Saves the best model locally and uploads it to W&B as an artifact, then cleans up the local file.
    """
    models_dir = "outputs/models"
    os.makedirs(models_dir, exist_ok=True)

    # Save the Model locally first so W&B can grab it
    model_filepath = f"{models_dir}/{base_name}.pkl"
    print(f"Saving model locally to {model_filepath}...")
    joblib.dump(model, model_filepath)

    # --- 3. UPLOAD MODEL TO W&B ---
    print("Uploading model to W&B Cloud...")
    model_artifact = wandb.Artifact(
        name=f"{base_name}_model",
        type="model",
        description="Trained  model"
    )
    model_artifact.add_file(model_filepath)
    wandb.log_artifact(model_artifact)

    if os.path.exists(model_filepath):
        os.remove(model_filepath)
        # Only remove directory if it's empty; use shutil.rmtree if cleanup needed
        try:
            os.rmdir(models_dir)
        except OSError:
            pass  # Directory not empty or other error - that's fine
        print("Deleted:", model_filepath)
    else:
        print("File not found:", model_filepath)


def save_submission(y_pred: pd.Series, file_name: str) -> None:
    """
    Generates Kaggle predictions and uploads the CSV to W&B.
    """
    submissions_dir = "outputs/submissions"
    os.makedirs(submissions_dir, exist_ok=True)

    output_path = f"{submissions_dir}/{file_name}.csv"

    print("Generating Kaggle submission...")

    submission_ids = range(len(y_pred))

    submission = pd.DataFrame({
        "ID": submission_ids,
        "POSITION": y_pred.astype(int)
    })

    submission.to_csv(output_path, index=False)
    print(f"Submission saved locally to: {output_path}")

    # UPLOAD CSV TO W&B ---
    print("Uploading Kaggle submission to W&B Cloud...")
    csv_artifact = wandb.Artifact(
        name=f"{file_name}_submission",
        type="predictions"
    )
    csv_artifact.add_file(output_path)
    wandb.log_artifact(csv_artifact)

    # --- 5. CLOSE THE W&B RUN ---
    wandb.finish()



#### Helper functions

In [9]:
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

## Models

In [10]:
from sklearn.neighbors import KNeighborsClassifier, KernelDensity
from sklearn.base import BaseEstimator, ClassifierMixin

#Usem Standard Scaled per models de distància
X_nonparam_train = X_train_scaled_standard
y_nonparam_train = y_train['position'].values

X_nonparam_val = X_val_scaled_standard
y_nonparam_val = y_val['position'].values

## K-Nearest Neighbors (KNN)

In [11]:
print("\n--- Running K-Nearest Neighbors (KNN) with GridSearchCV ---")
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

# Definim l'espai de cerca (k de 1 a 10, i els dos tipus de pesos)
param_grid_knn = {
    'n_neighbors': np.arange(1, 11), # Provem k de 1 fins a 20
    'weights': ['uniform', 'distance']
}

# Inicialitzem el model KNN base
knn_base = KNeighborsClassifier(metric='euclidean')

grid_search_knn = GridSearchCV(
    estimator=knn_base,
    param_grid=param_grid_knn,
    cv=cv, 
    scoring='f1_macro',
    n_jobs=-1, # Usa tots els processadors disponibles
    verbose=1
)

print("Searching for the best hyperparameters...")
grid_search_knn.fit(X_nonparam_train, y_nonparam_train)

best_knn = grid_search_knn.best_estimator_
best_params_knn = grid_search_knn.best_params_

print(f"Best parameters found: {best_params_knn}")
print(f"Best Cross-Validation F1-Macro: {grid_search_knn.best_score_:.4f}")

# Avaluem 
evaluate(
    base_name=f"KNN_Optimized_k{best_params_knn['n_neighbors']}_{best_params_knn['weights']}", 
    model=best_knn, 
    X_val=X_nonparam_val, 
    y_val=y_nonparam_val, 
    best_params=best_params_knn
)


--- Running K-Nearest Neighbors (KNN) with GridSearchCV ---
Searching for the best hyperparameters...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'n_neighbors': np.int64(3), 'weights': 'distance'}
Best Cross-Validation F1-Macro: 0.8392
Evaluating best model on unseen validation data...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\laura\_netrc.


Initializing Weights & Biases run...


wandb: Currently logged in as: laura-rebollo-crespo (laura-rebollo-crespo-universitat-polit-cnica-de-catalunya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Parzen Windows (Kernel Density Estimation Classifier)

In [12]:
print("\n--- Running Parzen Windows Classifier ---")

class ParzenWindowClassifier(BaseEstimator, ClassifierMixin):
    """
    Custom classifier applying Parzen Windows (KDE) for each class.
    Classifies a new sample by assigning it to the class with the 
    highest posterior probability (Log-Likelihood + Log-Prior).
    """
    def __init__(self, bandwidth=1.0, kernel='gaussian'):
        self.bandwidth = bandwidth
        self.kernel = kernel
        self.classes_ = None
        self.kdes_ = {}
        self.priors_ = {}

    def fit(self, X, y):
        # Assegurem que treballem amb Numpy Arrays
        X_arr = np.array(X)
        y_arr = np.array(y)
        
        self.classes_ = np.unique(y_arr)
        
        for c in self.classes_:
            X_c = X_arr[y_arr == c]
            
            self.priors_[c] = len(X_c) / len(X_arr)
            
            kde = KernelDensity(bandwidth=self.bandwidth, kernel=self.kernel)
            kde.fit(X_c)
            self.kdes_[c] = kde
            
        return self

    def predict(self, X):
        X_arr = np.array(X)
        log_probs = np.zeros((X_arr.shape[0], len(self.classes_)))
        
        for i, c in enumerate(self.classes_):
            # Bayes: log P(x|c) + log P(c)
            log_probs[:, i] = self.kdes_[c].score_samples(X_arr) + np.log(self.priors_[c])
            
        return self.classes_[np.argmax(log_probs, axis=1)]



parzen = ParzenWindowClassifier(bandwidth=1.0, kernel='gaussian')

parzen.fit(X_nonparam_train, y_nonparam_train)

evaluate(
    base_name="ParzenWindows_Gaussian_bw1", 
    model=parzen, 
    X_val=X_nonparam_val, 
    y_val=y_nonparam_val, 
    best_params={"bandwidth": 1.0, "kernel": "gaussian"}
)


--- Running Parzen Windows Classifier ---
Evaluating best model on unseen validation data...
Initializing Weights & Biases run...


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


val_accuracy,▁
val_f1_macro,▁
val_precision_macro,▁
val_recall_macro,▁
val_accuracy,0.84445
val_f1_macro,0.84613
val_precision_macro,0.85248
val_recall_macro,0.84483
